# B-TRUST Phase 70 — transformer and LLM models under the validity gates (run in Google Colab, GPU runtime)
**Preregistered here, before running.** Record the run date and SHA-256 of this notebook file in the results.

Hypotheses (mirroring H8 for new model classes; k = 5 corpora; exact one-sided permutation tests, α = 0.05):
- **H9a** Spearman ρ(G0 lift, mean transfer T) > 0 under fine-tuned `distilbert-base-uncased`.
- **H9b** G0 decisions (same thresholds as Phase 65) identical to TF-IDF for all five corpora under DistilBERT.
- **H9c, H9d** the same two tests under `roberta-base`.
- **H10 (exploratory-confirmatory)** a zero-shot LLM (Claude; record the exact model string) has lower G6 vote loss on YouLeQD 70 than
  the best constant (0.738), paired video-cluster bootstrap 10,000, seed 70, 95% CI upper bound < 0.

Fixed settings (no tuning): max_len 64, batch 16, lr 2e-5, 3 epochs, weight decay 0.01, seed 70, AdamW, linear schedule.
Leakage rule and targets identical to Phase 68. LLM targets: DASQ random 500 (seed 70), SCAR-E random 500 (seed 70), EDUQG all,
YouLeQD 70, LAK 40.

## 1. Setup — upload `BTRUST_Frozen_Supporting_Files_2026-11-07.zip` and `BTRUST_v2.1_*.zip` when prompted

In [ ]:
!pip -q install transformers==4.46.3 accelerate anthropic scikit-learn pandas openpyxl
import os, zipfile, glob, subprocess
from google.colab import files
up = files.upload()   # select the v1.0 frozen ZIP and the v2.1 ZIP
for z in up:
    zipfile.ZipFile(z).extractall('/content/in')
os.makedirs('/content/ex', exist_ok=True)
for z in glob.glob('/content/in/**/original_phase_archives/*.zip', recursive=True):
    n = os.path.basename(z)[:-4]; zipfile.ZipFile(z).extractall(f'/content/ex/{n}')
os.environ['BTRUST_EX'] = '/content/ex'
code_dir = glob.glob('/content/in/**/v2.0_phases65-68/code', recursive=True)[0]
os.chdir(code_dir); print(os.getcwd())
subprocess.run(['sh','fetch_and_prepare.sh'], check=True)
import sys; sys.path.insert(0, code_dir)

## 2. Corpora (identical loaders to Phases 65–69)

In [ ]:
import numpy as np, pandas as pd, itertools, torch, random
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import corpora as C
corp = {"EDUB":C.edub(),"DASQ":C.dasq(),"SCAR-I":C.scar_i(),"SCAR-E":C.scar_e(),"EDUQG":C.eduqg()}
yq, V = C.youleqd(); lk = C.lak()
print({k: len(v) for k, v in corp.items()}, len(yq), len(lk))

## 3. Fine-tuning helper

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def seed_all(s=70):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def fit_predict(name, Xtr, ytr, Xte_list):
    seed_all(); tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=6).to(dev)
    enc = tok(list(Xtr), truncation=True, max_length=64, padding=True, return_tensors='pt')
    y = torch.tensor(np.asarray(ytr) - 1)
    idx = np.arange(len(y)); opt = torch.optim.AdamW(m.parameters(), lr=2e-5, weight_decay=0.01)
    steps = 3 * int(np.ceil(len(y) / 16)); sch = get_linear_schedule_with_warmup(opt, 0, steps)
    m.train(); rg = np.random.default_rng(70)
    for ep in range(3):
        rg.shuffle(idx)
        for i in range(0, len(idx), 16):
            b = idx[i:i+16]
            out = m(input_ids=enc['input_ids'][b].to(dev), attention_mask=enc['attention_mask'][b].to(dev), labels=y[b].to(dev))
            out.loss.backward(); opt.step(); sch.step(); opt.zero_grad()
    m.eval(); preds = []
    for X in Xte_list:
        P = []
        for i in range(0, len(X), 64):
            e = tok(list(X[i:i+64]), truncation=True, max_length=64, padding=True, return_tensors='pt').to(dev)
            with torch.no_grad(): P.append(m(**e).logits.argmax(-1).cpu().numpy() + 1)
        preds.append(np.concatenate(P) if P else np.array([], int))
    return preds

## 4. Gates G0 and transfer for each transformer (H9a–H9d)

In [ ]:
def lookup_cv(key, y):
    p = np.zeros(len(y), int); maj = pd.Series(y).mode()[0]
    for a, b in StratifiedKFold(5, shuffle=True, random_state=65).split(key, y):
        t = pd.DataFrame({'k': key[a], 'y': y[a]}).groupby('k').y.agg(lambda s: s.mode()[0])
        p[b] = pd.Series(key[b]).map(t).fillna(maj).astype(int).to_numpy()
    return p
targets = {"DASQ": corp["DASQ"], "SCAR-E": corp["SCAR-E"], "EDUQG": corp["EDUQG"]}
tf = {"EDUB":"FAIL","DASQ":"PASS","SCAR-I":"PASS","SCAR-E":"PASS","EDUQG":"FAIL"}   # Phase 65 TF-IDF decisions
results = {}
for name in ["distilbert-base-uncased", "roberta-base"]:
    G, T = [], []
    for c, d in corp.items():
        X = d.text.to_numpy(); y = d.y.to_numpy(); p = np.zeros(len(y), int)
        for a, b in StratifiedKFold(5, shuffle=True, random_state=65).split(X, y):
            p[b] = fit_predict(name, X[a], y[a], [X[b]])[0]
        acc = (p == y).mean(); maj = pd.Series(y).value_counts(normalize=True).iloc[0]
        meta = (lookup_cv(d.meta.to_numpy(), y) == y).mean() if 'meta' in d else np.nan
        lift = acc - np.nanmax([maj, meta]); k = cohen_kappa_score(y, p)
        gate = "FAIL" if (lift < .05 or k < .20) else ("PASS" if (lift >= .15 and k >= .40) else "CAUTION")
        G.append(dict(model=name, corpus=c, cv_acc=acc, G0_lift=lift, kappa_cv=k, G0=gate, same_as_tfidf=gate == tf[c]))
        ts = set(d.n); tests, names = [], []
        for tn, t in targets.items():
            if tn == c or {c, tn} == {"SCAR-I", "SCAR-E"}: continue
            keep = ~t.n.isin(ts); tests.append(t.text[keep].to_numpy()); names.append((tn, t.y[keep].to_numpy()))
        P = fit_predict(name, X, y, tests + [yq.text.to_numpy(), lk.text.to_numpy()])
        accs = {tn: (pp == yt).mean() for (tn, yt), pp in zip(names, P[:len(names)])}
        accs["YLQ70"] = (P[-2] == yq.y.to_numpy()).mean(); accs["LAK40"] = np.mean([a in s for a, s in zip(P[-1], lk.refs)])
        T.append(dict(model=name, train=c, **accs, T_mean=np.mean(list(accs.values())), G0_lift=lift))
    G = pd.DataFrame(G); T = pd.DataFrame(T); results[name] = (G, T)
    obs = spearmanr(T.G0_lift, T.T_mean).statistic
    pv = np.mean([spearmanr(T.G0_lift, np.array(q)).statistic >= obs - 1e-12 for q in itertools.permutations(T.T_mean)])
    print(name, f"rho={obs:.3f} exact one-sided p={pv:.4f}  identical G0 decisions={bool(G.same_as_tfidf.all())}")
    display(G.round(4)); display(T.round(4))
    G.to_csv(f"p70_G0_{name.split('-')[0]}.csv", index=False); T.to_csv(f"p70_transfer_{name.split('-')[0]}.csv", index=False)

## 5. Zero-shot LLM on the human-judged targets (H10). Enter your own API key; cost is roughly 2,000 short calls.

In [ ]:
import anthropic, getpass, json, re, time
client = anthropic.Anthropic(api_key=getpass.getpass('Anthropic API key: '))
MODEL = 'claude-sonnet-5-5'   # record the exact model string used
PROMPT = ('Classify the cognitive level required to answer the question below using the original Bloom taxonomy. '
          'Answer with one word only: Knowledge, Comprehension, Application, Analysis, Evaluation, or Synthesis.\n\nQuestion: ')
MAP = {"knowledge":1,"comprehension":2,"application":3,"analysis":4,"evaluation":5,"synthesis":6}
def llm(t):
    for _ in range(5):
        try:
            r = client.messages.create(model=MODEL, max_tokens=5, temperature=0, messages=[{"role":"user","content":PROMPT+t}])
            w = re.sub(r'[^a-z]', '', r.content[0].text.lower()); return MAP.get(w, 0)
        except Exception as e: time.sleep(5)
    return 0
rg = np.random.default_rng(70)
sets = {"DASQ": corp["DASQ"].iloc[rg.choice(len(corp["DASQ"]), 500, replace=False)],
        "SCAR-E": corp["SCAR-E"].iloc[rg.choice(len(corp["SCAR-E"]), 500, replace=False)], "EDUQG": corp["EDUQG"]}
out = {}
for k, d in sets.items():
    p = np.array([llm(t) for t in d.text]); out[k] = (p == d.y.to_numpy()).mean(); print(k, out[k])
pq = np.array([llm(t) for t in yq.text]); out["YLQ70"] = (pq == yq.y.to_numpy()).mean()
pl = np.array([llm(t) for t in lk.text]); out["LAK40"] = np.mean([a in s for a, s in zip(pl, lk.refs)])
vl = np.nanmean(np.where(np.isnan(V), 1.0, (V != pq[:, None]).astype(float)), 1)
const = np.nanmean(np.where(np.isnan(V), 1.0, (V != 2).astype(float)), 1)   # best constant: comprehension
d = vl - const; g = pd.Series(d).groupby(yq.vid.values); s = g.sum().to_numpy(); c = g.size().to_numpy(); rb = np.random.default_rng(70)
bs = [s[i].sum() / c[i].sum() for i in (rb.integers(0, len(s), len(s)) for _ in range(10000))]
print(out); print(f"H10: LLM vote loss {vl.mean():.4f} vs constant {const.mean():.4f}; diff {d.mean():+.4f}, CI [{np.percentile(bs,2.5):+.4f}, {np.percentile(bs,97.5):+.4f}]; supported={np.percentile(bs,97.5)<0}")
pd.DataFrame([dict(model=MODEL, **out, ylq_voteloss=vl.mean(), h10_ci_hi=np.percentile(bs,97.5))]).to_csv("p70_llm.csv", index=False)

## 6. Download all `p70_*.csv` files and add them to the archive with this notebook's SHA-256.

In [ ]:
import glob; [files.download(f) for f in glob.glob('p70_*.csv')]